In [0]:
dbutils.library.restartPython()

In [0]:
import json
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_s3_parquet, write_raw

In [0]:
# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("s3-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================

with open(f"{config_path}", "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

aws_access_key_id = dbutils.secrets.get(scope='s3-ingestion-dev', key=source_config['aws_access_key_id'])
aws_secret_access_key = dbutils.secrets.get(scope='s3-ingestion-dev', key=source_config['aws_secret_access_key'])

# ============================================================
# 5. READ FROM S3 SERVER
# ============================================================
df = read_s3_parquet(spark,aws_access_key_id,aws_secret_access_key,source_config["path"])
logger.info("Data Succesfully Read")

# ============================================================
# 6. TARGET PATH
# ============================================================
target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"

# ============================================================
# 7. WRITE RAW
# ============================================================
logger.warning("Writing the in the target %s", target_path)
write_raw(df,target_path,target_config["file_format"],target_config["mode"])
logger.info("Data Succesfully written")
